# Merchants - Bronze

- Extract `mcc` (zero-padded to 4 digits), `city` and `country` from the nested columns
- Trim merchant names and parse `onboarded_on`
- Keep one row per `merchant_id`, preferring names that don't end in `(old)`

**Reads:** `bronze.bronze_merchants`  
**Writes:** `silver.silver_merchants` (overwrite)

# Importing Function

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Merchant DataFrame

In [ ]:
merchant_df = spark.read.table("bronze.bronze_merchants")

## Parsing Requested Attributes (MCC, City, Country) and Removing Old Data

In [ ]:
merch_c = (
    merchant_df.withColumn("mcc", F.lpad(F.trim("category.mcc"), 4, "0"))
    .withColumn("merchant_name", F.trim("merchant_name"))
    .withColumn("city", F.col("address.city"))
    .withColumn("country", F.col("address.country"))
    .withColumn("is_old", F.col("merchant_name").endswith("(old)"))
    .withColumn("onboarded_on", F.to_date("onboarded_on", "yyyy-MM-dd"))
    .withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("merchant_id").orderBy("is_old", "merchant_name")
        ),
    )
    .filter("_rn = 1")
    .drop("_rn", "is_old")
)
print(merchant_df.count(), merch_c.count())

In [ ]:
display(merch_c)

## Processing to Validate Staging Layer

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
merch_c.write.mode("overwrite").saveAsTable("silver.silver_merchants")